In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

from sklearn.metrics import precision_score, recall_score, f1_score

print("Libraries loaded successfully")

Libraries loaded successfully


In [55]:
# Load the original dataset
df = pd.read_csv("teen_phone_addiction_dataset.csv")

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

# Preview first 5 records
display(df.head())

Dataset loaded successfully
Dataset shape: (3000, 25)


,ID,Name,Age,Gender,Location,School_Grade,Daily_Usage_Hours,Sleep_Hours,Academic_Performance,Social_Interactions,...,Screen_Time_Before_Bed,Phone_Checks_Per_Day,Apps_Used_Daily,Time_on_Social_Media,Time_on_Gaming,Time_on_Education,Phone_Usage_Purpose,Family_Communication,Weekend_Usage_Hours,Addiction_Level
0,1,Shannon Francis,13,Female,Hansonfort,9th,4.0,6.1,78,5,...,1.4,86,19,3.6,1.7,1.2,Browsing,4,8.7,10.0
1,2,Scott Rodriguez,17,Female,Theodorefort,7th,5.5,6.5,70,5,...,0.9,96,9,1.1,4.0,1.8,Browsing,2,5.3,10.0
2,3,Adrian Knox,13,Other,Lindseystad,11th,5.8,5.5,93,8,...,0.5,137,8,0.3,1.5,0.4,Education,6,5.7,9.2
3,4,Brittany Hamilton,18,Female,West Anthony,12th,3.1,3.9,78,8,...,1.4,128,7,3.1,1.6,0.8,Social Media,8,3.0,9.8
4,5,Steven Smith,14,Other,Port Lindsaystad,9th,2.5,6.7,56,4,...,1.0,96,20,2.6,0.9,1.1,Gaming,10,3.7,8.6


In [56]:
# ==========================================
# STEP 3: BASIC DATASET VALIDATION
# ==========================================

print("===== DATASET INFORMATION =====")
df.info()

print("\n===== MISSING VALUES =====")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

print("\n===== DUPLICATE ROWS =====")
print("Number of duplicate rows:", df.duplicated().sum())

print("\n===== NUMERICAL DATA SUMMARY =====")
display(df.describe())

===== DATASET INFORMATION =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ID                      3000 non-null   int64  
 1   Name                    3000 non-null   object 
 2   Age                     3000 non-null   int64  
 3   Gender                  3000 non-null   object 
 4   Location                3000 non-null   object 
 5   School_Grade            3000 non-null   object 
 6   Daily_Usage_Hours       3000 non-null   float64
 7   Sleep_Hours             3000 non-null   float64
 8   Academic_Performance    3000 non-null   int64  
 9   Social_Interactions     3000 non-null   int64  
 10  Exercise_Hours          3000 non-null   float64
 11  Anxiety_Level           3000 non-null   int64  
 12  Depression_Level        3000 non-null   int64  
 13  Self_Esteem             3000 non-null   int64  
 14  Parental

,ID,Age,Daily_Usage_Hours,Sleep_Hours,Academic_Performance,Social_Interactions,Exercise_Hours,Anxiety_Level,Depression_Level,Self_Esteem,Parental_Control,Screen_Time_Before_Bed,Phone_Checks_Per_Day,Apps_Used_Daily,Time_on_Social_Media,Time_on_Gaming,Time_on_Education,Family_Communication,Weekend_Usage_Hours,Addiction_Level
count,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,1500.500000,15.969667,5.020667,6.489767,74.947333,5.097667,1.040667,5.590000,5.460333,5.546333,0.507333,1.006733,83.093000,12.609333,2.499233,1.525267,1.016333,5.459667,6.015100,8.881900
std,866.169729,1.989489,1.956501,1.490713,14.684156,3.139333,0.734620,2.890678,2.871557,2.860754,0.500030,0.492878,37.747044,4.611486,0.988201,0.932701,0.648341,2.864572,2.014776,1.609598
min,1.000000,13.000000,0.000000,3.000000,50.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,20.000000,5.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000
25%,750.750000,14.000000,3.700000,5.500000,62.000000,2.000000,0.500000,3.000000,3.000000,3.000000,0.000000,0.700000,51.000000,9.000000,1.800000,0.800000,0.500000,3.000000,4.700000,8.000000
50%,1500.500000,16.000000,5.000000,6.500000,75.000000,5.000000,1.000000,6.000000,5.000000,6.000000,1.000000,1.000000,82.000000,13.000000,2.500000,1.500000,1.000000,5.000000,6.000000,10.000000
75%,2250.250000,18.000000,6.400000,7.500000,88.000000,8.000000,1.500000,8.000000,8.000000,8.000000,1.000000,1.400000,115.250000,17.000000,3.200000,2.200000,1.500000,8.000000,7.400000,10.000000
max,3000.000000,19.000000,11.500000,10.000000,100.000000,10.000000,4.000000,10.000000,10.000000,10.000000,1.000000,2.600000,150.000000,20.000000,5.000000,4.000000,3.000000,10.000000,14.000000,10.000000


In [57]:
# ==========================================
# STEP 4: SELECT BEHAVIORAL FEATURES
# ==========================================

behavioral_features = [
    "Daily_Usage_Hours",
    "Sleep_Hours",
    "Academic_Performance",
    "Social_Interactions",
    "Exercise_Hours",
    "Screen_Time_Before_Bed",
    "Phone_Checks_Per_Day",
    "Apps_Used_Daily",
    "Time_on_Social_Media",
    "Time_on_Gaming",
    "Time_on_Education",
    "Family_Communication",
    "Weekend_Usage_Hours"
]

# Create Function 2 dataset
X = df[behavioral_features].copy()

print("Behavioral dataset created successfully")
print("Shape:", X.shape)

print("\nSelected behavioral features:")
for feature in behavioral_features:
    print("-", feature)

print("\n===== FEATURE RANGES =====")
feature_ranges = pd.DataFrame({
    "Min": X.min(),
    "Max": X.max(),
    "Mean": X.mean(),
    "Median": X.median()
})

display(feature_ranges)

Behavioral dataset created successfully
Shape: (3000, 13)

Selected behavioral features:
- Daily_Usage_Hours
- Sleep_Hours
- Academic_Performance
- Social_Interactions
- Exercise_Hours
- Screen_Time_Before_Bed
- Phone_Checks_Per_Day
- Apps_Used_Daily
- Time_on_Social_Media
- Time_on_Gaming
- Time_on_Education
- Family_Communication
- Weekend_Usage_Hours

===== FEATURE RANGES =====


,Min,Max,Mean,Median
Daily_Usage_Hours,0.0,11.5,5.020667,5.0
Sleep_Hours,3.0,10.0,6.489767,6.5
Academic_Performance,50.0,100.0,74.947333,75.0
Social_Interactions,0.0,10.0,5.097667,5.0
Exercise_Hours,0.0,4.0,1.040667,1.0
Screen_Time_Before_Bed,0.0,2.6,1.006733,1.0
Phone_Checks_Per_Day,20.0,150.0,83.093000,82.0
Apps_Used_Daily,5.0,20.0,12.609333,13.0
Time_on_Social_Media,0.0,5.0,2.499233,2.5
Time_on_Gaming,0.0,4.0,1.525267,1.5


In [58]:
from sklearn.model_selection import train_test_split

In [59]:
# ==========================================
# STEP 5: TRAIN / TEST SPLIT
# ==========================================

X_train, X_test = train_test_split(
    X,
    test_size=0.20,
    random_state=42
)

print("Train/Test split completed successfully")

print("\nFull dataset shape :", X.shape)
print("Training set shape:", X_train.shape)
print("Testing set shape :", X_test.shape)

print("\nTraining percentage:",
      round(len(X_train) / len(X) * 100, 2), "%")

print("Testing percentage:",
      round(len(X_test) / len(X) * 100, 2), "%")

Train/Test split completed successfully

Full dataset shape : (3000, 13)
Training set shape: (2400, 13)
Testing set shape : (600, 13)

Training percentage: 80.0 %
Testing percentage: 20.0 %


In [60]:
# ==========================================
# STEP 6: FEATURE SCALING
# ==========================================

# Create the scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same fitted scaler to transform test data
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully")

print("\nTraining scaled shape:", X_train_scaled.shape)
print("Testing scaled shape :", X_test_scaled.shape)

# Check the scaled training data
print("\nMean of scaled training features:")
print(np.round(X_train_scaled.mean(axis=0), 2))

print("\nStandard deviation of scaled training features:")
print(np.round(X_train_scaled.std(axis=0), 2))

Feature scaling completed successfully

Training scaled shape: (2400, 13)
Testing scaled shape : (600, 13)

Mean of scaled training features:
[ 0.  0. -0.  0. -0. -0. -0.  0.  0. -0. -0.  0.  0.]

Standard deviation of scaled training features:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [61]:
# ==========================================
# STEP 7: TRAIN ISOLATION FOREST
# ==========================================

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.10,
    random_state=42
)

# Train the model using training data
iso_forest.fit(X_train_scaled)

# Predictions on test data
#  1 = normal
# -1 = anomaly
iso_predictions = iso_forest.predict(X_test_scaled)

# Anomaly scores
iso_scores = iso_forest.decision_function(X_test_scaled)

print("Isolation Forest trained successfully")

# Count predictions
normal_count = np.sum(iso_predictions == 1)
anomaly_count = np.sum(iso_predictions == -1)

print("\n===== ISOLATION FOREST RESULTS =====")
print("Normal records :", normal_count)
print("Anomaly records:", anomaly_count)

print("\nNormal percentage:",
      round(normal_count / len(iso_predictions) * 100, 2), "%")

print("Anomaly percentage:",
      round(anomaly_count / len(iso_predictions) * 100, 2), "%")

print("\nFirst 10 predictions:")
print(iso_predictions[:10])

print("\nFirst 10 anomaly scores:")
print(np.round(iso_scores[:10], 4))

Isolation Forest trained successfully

===== ISOLATION FOREST RESULTS =====
Normal records : 526
Anomaly records: 74

Normal percentage: 87.67 %
Anomaly percentage: 12.33 %

First 10 predictions:
[ 1  1  1  1  1  1 -1  1 -1  1]

First 10 anomaly scores:
[ 0.0692  0.082   0.0168  0.04    0.0669  0.0357 -0.0138  0.0382 -0.0131
  0.0745]


In [62]:
# ==========================================
# STEP 8: TRAIN ONE-CLASS SVM
# ==========================================

oc_svm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.10
)

# Train model
oc_svm.fit(X_train_scaled)

# Predictions on test data
#  1 = normal
# -1 = anomaly
svm_predictions = oc_svm.predict(X_test_scaled)

# Anomaly scores
svm_scores = oc_svm.decision_function(X_test_scaled)

print("One-Class SVM trained successfully")

# Count predictions
svm_normal_count = np.sum(svm_predictions == 1)
svm_anomaly_count = np.sum(svm_predictions == -1)

print("\n===== ONE-CLASS SVM RESULTS =====")
print("Normal records :", svm_normal_count)
print("Anomaly records:", svm_anomaly_count)

print("\nNormal percentage:",
      round(svm_normal_count / len(svm_predictions) * 100, 2), "%")

print("Anomaly percentage:",
      round(svm_anomaly_count / len(svm_predictions) * 100, 2), "%")

print("\nFirst 10 predictions:")
print(svm_predictions[:10])

print("\nFirst 10 anomaly scores:")
print(np.round(svm_scores[:10], 4))

One-Class SVM trained successfully

===== ONE-CLASS SVM RESULTS =====
Normal records : 529
Anomaly records: 71

Normal percentage: 88.17 %
Anomaly percentage: 11.83 %

First 10 predictions:
[ 1  1  1  1  1  1 -1  1 -1  1]

First 10 anomaly scores:
[12.9992 12.9418  2.2948  5.675  10.2269  2.9235 -2.8821  4.8723 -2.2645
 15.0737]


In [63]:
# ==========================================
# STEP 9: CREATE CONTROLLED SYNTHETIC ANOMALIES
# ==========================================

np.random.seed(42)

# Keep 540 original test records as normal evaluation cases
normal_eval = X_test.sample(n=540, random_state=42).copy()
normal_eval["ground_truth_anomaly"] = 0
normal_eval["anomaly_type"] = "normal"

# Select 60 different records as bases for synthetic anomalies
remaining_indices = X_test.index.difference(normal_eval.index)
synthetic = X_test.loc[remaining_indices].copy()

# We have exactly 60 records
synthetic = synthetic.iloc[:60].copy()

# ------------------------------------------------
# Pattern 1: Excessive phone usage + low sleep
# ------------------------------------------------
idx1 = synthetic.index[:20]

synthetic.loc[idx1, "Daily_Usage_Hours"] = 10.5
synthetic.loc[idx1, "Sleep_Hours"] = 3.5
synthetic.loc[idx1, "Phone_Checks_Per_Day"] = 145

# ------------------------------------------------
# Pattern 2: Heavy night/social media usage
# ------------------------------------------------
idx2 = synthetic.index[20:40]

synthetic.loc[idx2, "Screen_Time_Before_Bed"] = 2.5
synthetic.loc[idx2, "Time_on_Social_Media"] = 4.8
synthetic.loc[idx2, "Sleep_Hours"] = 3.5

# ------------------------------------------------
# Pattern 3: Heavy weekend/gaming usage
# ------------------------------------------------
idx3 = synthetic.index[40:60]

synthetic.loc[idx3, "Weekend_Usage_Hours"] = 13.5
synthetic.loc[idx3, "Time_on_Gaming"] = 3.8
synthetic.loc[idx3, "Daily_Usage_Hours"] = 10.5

# Ground-truth labels
synthetic["ground_truth_anomaly"] = 1

synthetic.loc[idx1, "anomaly_type"] = "high_usage_low_sleep"
synthetic.loc[idx2, "anomaly_type"] = "night_social_low_sleep"
synthetic.loc[idx3, "anomaly_type"] = "weekend_gaming_high_usage"

# Combine normal + synthetic records
evaluation_data = pd.concat(
    [normal_eval, synthetic],
    axis=0
).sample(frac=1, random_state=42)

print("Controlled evaluation dataset created successfully")

print("\nTotal evaluation records:", len(evaluation_data))
print("Normal records:",
      (evaluation_data["ground_truth_anomaly"] == 0).sum())
print("Synthetic anomaly records:",
      (evaluation_data["ground_truth_anomaly"] == 1).sum())

print("\n===== ANOMALY TYPES =====")
print(evaluation_data["anomaly_type"].value_counts())

Controlled evaluation dataset created successfully

Total evaluation records: 600
Normal records: 540
Synthetic anomaly records: 60

===== ANOMALY TYPES =====
anomaly_type
normal                       540
night_social_low_sleep        20
weekend_gaming_high_usage     20
high_usage_low_sleep          20
Name: count, dtype: int64


In [64]:
# ==========================================
# STEP 10: CALCULATE DATA-DRIVEN THRESHOLDS
# ==========================================

thresholds = {
    "Daily_Usage_Hours_high":
        X_train["Daily_Usage_Hours"].quantile(0.95),

    "Sleep_Hours_low":
        X_train["Sleep_Hours"].quantile(0.05),

    "Phone_Checks_Per_Day_high":
        X_train["Phone_Checks_Per_Day"].quantile(0.95),

    "Screen_Time_Before_Bed_high":
        X_train["Screen_Time_Before_Bed"].quantile(0.95),

    "Time_on_Social_Media_high":
        X_train["Time_on_Social_Media"].quantile(0.95),

    "Time_on_Gaming_high":
        X_train["Time_on_Gaming"].quantile(0.95),

    "Weekend_Usage_Hours_high":
        X_train["Weekend_Usage_Hours"].quantile(0.95)
}

print("===== DATA-DRIVEN ANOMALY THRESHOLDS =====")

for feature, value in thresholds.items():
    print(f"{feature}: {value:.2f}")

===== DATA-DRIVEN ANOMALY THRESHOLDS =====
Daily_Usage_Hours_high: 8.20
Sleep_Hours_low: 4.00
Phone_Checks_Per_Day_high: 143.00
Screen_Time_Before_Bed_high: 1.80
Time_on_Social_Media_high: 4.20
Time_on_Gaming_high: 3.10
Weekend_Usage_Hours_high: 9.30


In [65]:
# ==========================================
# STEP 11: CREATE DATA-DRIVEN
# SYNTHETIC ANOMALY DATASET
# ==========================================

np.random.seed(42)

# Select 60 records from the test set
synthetic_indices = X_test.sample(
    n=60,
    random_state=42
).index

synthetic_anomalies = X_test.loc[
    synthetic_indices
].copy()

# Split into 3 anomaly groups
idx1 = synthetic_anomalies.index[:20]
idx2 = synthetic_anomalies.index[20:40]
idx3 = synthetic_anomalies.index[40:60]


# ------------------------------------------
# Pattern 1:
# High phone usage + low sleep + many checks
# ------------------------------------------

synthetic_anomalies.loc[
    idx1, "Daily_Usage_Hours"
] = min(
    thresholds["Daily_Usage_Hours_high"] + 1.0,
    X_train["Daily_Usage_Hours"].max()
)

synthetic_anomalies.loc[
    idx1, "Sleep_Hours"
] = max(
    thresholds["Sleep_Hours_low"] - 0.5,
    X_train["Sleep_Hours"].min()
)

synthetic_anomalies.loc[
    idx1, "Phone_Checks_Per_Day"
] = min(
    thresholds["Phone_Checks_Per_Day_high"] + 5,
    X_train["Phone_Checks_Per_Day"].max()
)


# ------------------------------------------
# Pattern 2:
# High night screen time + social media
# + low sleep
# ------------------------------------------

synthetic_anomalies.loc[
    idx2, "Screen_Time_Before_Bed"
] = min(
    thresholds["Screen_Time_Before_Bed_high"] + 0.4,
    X_train["Screen_Time_Before_Bed"].max()
)

synthetic_anomalies.loc[
    idx2, "Time_on_Social_Media"
] = min(
    thresholds["Time_on_Social_Media_high"] + 0.5,
    X_train["Time_on_Social_Media"].max()
)

synthetic_anomalies.loc[
    idx2, "Sleep_Hours"
] = max(
    thresholds["Sleep_Hours_low"] - 0.5,
    X_train["Sleep_Hours"].min()
)


# ------------------------------------------
# Pattern 3:
# High weekend usage + gaming
# + high daily usage
# ------------------------------------------

synthetic_anomalies.loc[
    idx3, "Weekend_Usage_Hours"
] = min(
    thresholds["Weekend_Usage_Hours_high"] + 1.5,
    X_train["Weekend_Usage_Hours"].max()
)

synthetic_anomalies.loc[
    idx3, "Time_on_Gaming"
] = min(
    thresholds["Time_on_Gaming_high"] + 0.5,
    X_train["Time_on_Gaming"].max()
)

synthetic_anomalies.loc[
    idx3, "Daily_Usage_Hours"
] = min(
    thresholds["Daily_Usage_Hours_high"] + 1.0,
    X_train["Daily_Usage_Hours"].max()
)


# Add labels
synthetic_anomalies["ground_truth_anomaly"] = 1

synthetic_anomalies.loc[
    idx1, "anomaly_type"
] = "high_usage_low_sleep"

synthetic_anomalies.loc[
    idx2, "anomaly_type"
] = "night_social_low_sleep"

synthetic_anomalies.loc[
    idx3, "anomaly_type"
] = "weekend_gaming_high_usage"


print("Synthetic anomalies generated successfully")

print("\nTotal synthetic anomalies:",
      len(synthetic_anomalies))

print("\n===== ANOMALY TYPES =====")
print(
    synthetic_anomalies[
        "anomaly_type"
    ].value_counts()
)

display(synthetic_anomalies.head())

Synthetic anomalies generated successfully

Total synthetic anomalies: 60

===== ANOMALY TYPES =====
anomaly_type
high_usage_low_sleep         20
night_social_low_sleep       20
weekend_gaming_high_usage    20
Name: count, dtype: int64


,Daily_Usage_Hours,Sleep_Hours,Academic_Performance,Social_Interactions,Exercise_Hours,Screen_Time_Before_Bed,Phone_Checks_Per_Day,Apps_Used_Daily,Time_on_Social_Media,Time_on_Gaming,Time_on_Education,Family_Communication,Weekend_Usage_Hours,ground_truth_anomaly,anomaly_type
2986,9.2,3.5,81,9,1.4,0.4,148,12,2.0,1.7,1.5,3,5.9,1,high_usage_low_sleep
410,9.2,3.5,87,9,1.0,0.0,148,5,1.6,1.3,1.0,6,6.0,1,high_usage_low_sleep
1920,9.2,3.5,93,9,0.7,1.1,148,15,3.6,1.2,1.7,3,4.7,1,high_usage_low_sleep
2724,9.2,3.5,71,1,0.0,1.8,148,7,2.3,2.9,2.1,4,10.5,1,high_usage_low_sleep
2677,9.2,3.5,95,7,0.8,1.9,148,9,2.8,1.0,1.9,1,6.5,1,high_usage_low_sleep


In [66]:
# ==========================================
# STEP 12: VALIDATE SYNTHETIC ANOMALY DATA
# ==========================================

print("===== 1. BASIC VALIDATION =====")

print("Total synthetic records:", len(synthetic_anomalies))

missing_values = synthetic_anomalies[
    behavioral_features
].isnull().sum().sum()

print("Missing values:", missing_values)

duplicate_rows = synthetic_anomalies[
    behavioral_features
].duplicated().sum()

print("Duplicate rows:", duplicate_rows)


# ==========================================
# 2. RANGE VALIDATION
# Check whether synthetic values remain
# within observed training-data ranges
# ==========================================

print("\n===== 2. RANGE VALIDATION =====")

range_results = {}

for feature in behavioral_features:

    train_min = X_train[feature].min()
    train_max = X_train[feature].max()

    valid = synthetic_anomalies[feature].between(
        train_min,
        train_max
    ).all()

    range_results[feature] = valid

    print(
        f"{feature}: "
        f"{'PASS' if valid else 'FAIL'}"
    )


# ==========================================
# 3. ANOMALY RULE VALIDATION
# ==========================================

print("\n===== 3. ANOMALY RULE VALIDATION =====")

# Pattern 1
pattern1_valid = (
    (
        synthetic_anomalies.loc[
            idx1, "Daily_Usage_Hours"
        ] > thresholds["Daily_Usage_Hours_high"]
    )
    &
    (
        synthetic_anomalies.loc[
            idx1, "Sleep_Hours"
        ] < thresholds["Sleep_Hours_low"]
    )
    &
    (
        synthetic_anomalies.loc[
            idx1, "Phone_Checks_Per_Day"
        ] > thresholds["Phone_Checks_Per_Day_high"]
    )
).all()


# Pattern 2
pattern2_valid = (
    (
        synthetic_anomalies.loc[
            idx2, "Screen_Time_Before_Bed"
        ] > thresholds["Screen_Time_Before_Bed_high"]
    )
    &
    (
        synthetic_anomalies.loc[
            idx2, "Time_on_Social_Media"
        ] > thresholds["Time_on_Social_Media_high"]
    )
    &
    (
        synthetic_anomalies.loc[
            idx2, "Sleep_Hours"
        ] < thresholds["Sleep_Hours_low"]
    )
).all()


# Pattern 3
pattern3_valid = (
    (
        synthetic_anomalies.loc[
            idx3, "Weekend_Usage_Hours"
        ] > thresholds["Weekend_Usage_Hours_high"]
    )
    &
    (
        synthetic_anomalies.loc[
            idx3, "Time_on_Gaming"
        ] > thresholds["Time_on_Gaming_high"]
    )
    &
    (
        synthetic_anomalies.loc[
            idx3, "Daily_Usage_Hours"
        ] > thresholds["Daily_Usage_Hours_high"]
    )
).all()


print(
    "High usage + low sleep:",
    "PASS" if pattern1_valid else "FAIL"
)

print(
    "Night/social + low sleep:",
    "PASS" if pattern2_valid else "FAIL"
)

print(
    "Weekend/gaming + high usage:",
    "PASS" if pattern3_valid else "FAIL"
)


# ==========================================
# 4. LABEL VALIDATION
# ==========================================

print("\n===== 4. LABEL VALIDATION =====")

label_valid = (
    synthetic_anomalies[
        "ground_truth_anomaly"
    ] == 1
).all()

print(
    "All synthetic records labelled anomaly:",
    "PASS" if label_valid else "FAIL"
)


# ==========================================
# 5. FINAL VALIDATION RESULT
# ==========================================

all_ranges_valid = all(range_results.values())

overall_valid = (
    missing_values == 0
    and all_ranges_valid
    and pattern1_valid
    and pattern2_valid
    and pattern3_valid
    and label_valid
)

print("\n===== FINAL RESULT =====")

if overall_valid:
    print("SYNTHETIC DATA VALIDATION: PASS")
else:
    print("SYNTHETIC DATA VALIDATION: FAIL")

===== 1. BASIC VALIDATION =====
Total synthetic records: 60
Missing values: 0
Duplicate rows: 0

===== 2. RANGE VALIDATION =====
Daily_Usage_Hours: PASS
Sleep_Hours: PASS
Academic_Performance: PASS
Social_Interactions: PASS
Exercise_Hours: PASS
Screen_Time_Before_Bed: PASS
Phone_Checks_Per_Day: PASS
Apps_Used_Daily: PASS
Time_on_Social_Media: PASS
Time_on_Gaming: PASS
Time_on_Education: PASS
Family_Communication: PASS
Weekend_Usage_Hours: PASS

===== 3. ANOMALY RULE VALIDATION =====
High usage + low sleep: PASS
Night/social + low sleep: PASS
Weekend/gaming + high usage: PASS

===== 4. LABEL VALIDATION =====
All synthetic records labelled anomaly: PASS

===== FINAL RESULT =====
SYNTHETIC DATA VALIDATION: PASS


In [67]:
# ==========================================
# STEP 13: BUILD CONTROLLED EVALUATION SET
# ==========================================

# Remove the 60 records that were selected for
# synthetic anomaly generation
reference_indices = X_test.index.difference(
    synthetic_indices
)

# 540 untouched original test records
reference_data = X_test.loc[
    reference_indices
].copy()

reference_data["ground_truth_anomaly"] = 0
reference_data["anomaly_type"] = "non_injected_original"


# Keep validated synthetic anomalies
synthetic_eval = synthetic_anomalies.copy()


# Combine both groups
evaluation_data = pd.concat(
    [reference_data, synthetic_eval],
    axis=0
)

# Shuffle evaluation dataset
evaluation_data = evaluation_data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# Check final dataset
print("Controlled evaluation dataset created successfully")

print("\nTotal records:", len(evaluation_data))

print("\n===== CLASS DISTRIBUTION =====")
print(
    evaluation_data[
        "ground_truth_anomaly"
    ].value_counts().sort_index()
)

print("\n===== ANOMALY TYPES =====")
print(
    evaluation_data[
        "anomaly_type"
    ].value_counts()
)

print("\nMissing values:",
      evaluation_data[
          behavioral_features
      ].isnull().sum().sum())

Controlled evaluation dataset created successfully

Total records: 600

===== CLASS DISTRIBUTION =====
ground_truth_anomaly
0    540
1     60
Name: count, dtype: int64

===== ANOMALY TYPES =====
anomaly_type
non_injected_original        540
night_social_low_sleep        20
weekend_gaming_high_usage     20
high_usage_low_sleep          20
Name: count, dtype: int64

Missing values: 0


In [68]:
# ==========================================
# STEP 14: EVALUATE BOTH ANOMALY MODELS
# ==========================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    accuracy_score
)

# ------------------------------------------
# 1. Separate features and labels
# ------------------------------------------

X_eval = evaluation_data[behavioral_features].copy()

y_true = evaluation_data[
    "ground_truth_anomaly"
].values


# ------------------------------------------
# 2. Scale using EXISTING scaler
# Do NOT fit scaler again
# ------------------------------------------

X_eval_scaled = scaler.transform(X_eval)


# ------------------------------------------
# 3. Isolation Forest predictions
# ------------------------------------------

iso_raw_predictions = iso_forest.predict(
    X_eval_scaled
)

# Convert:
# -1 = anomaly -> 1
#  1 = normal  -> 0

iso_y_pred = (
    iso_raw_predictions == -1
).astype(int)


# ------------------------------------------
# 4. One-Class SVM predictions
# ------------------------------------------

svm_raw_predictions = oc_svm.predict(
    X_eval_scaled
)

svm_y_pred = (
    svm_raw_predictions == -1
).astype(int)


# ------------------------------------------
# 5. Calculate metrics
# ------------------------------------------

iso_precision = precision_score(
    y_true, iso_y_pred, zero_division=0
)

iso_recall = recall_score(
    y_true, iso_y_pred, zero_division=0
)

iso_f1 = f1_score(
    y_true, iso_y_pred, zero_division=0
)

iso_accuracy = accuracy_score(
    y_true, iso_y_pred
)


svm_precision = precision_score(
    y_true, svm_y_pred, zero_division=0
)

svm_recall = recall_score(
    y_true, svm_y_pred, zero_division=0
)

svm_f1 = f1_score(
    y_true, svm_y_pred, zero_division=0
)

svm_accuracy = accuracy_score(
    y_true, svm_y_pred
)


# ------------------------------------------
# 6. Display comparison
# ------------------------------------------

results = pd.DataFrame({
    "Model": [
        "Isolation Forest",
        "One-Class SVM"
    ],

    "Accuracy": [
        iso_accuracy,
        svm_accuracy
    ],

    "Precision": [
        iso_precision,
        svm_precision
    ],

    "Recall": [
        iso_recall,
        svm_recall
    ],

    "F1 Score": [
        iso_f1,
        svm_f1
    ]
})

print("===== MODEL PERFORMANCE =====")

display(
    results.round(4)
)


# ------------------------------------------
# 7. Confusion matrices
# ------------------------------------------

print("\n===== ISOLATION FOREST CONFUSION MATRIX =====")
print(confusion_matrix(y_true, iso_y_pred))

print("\n===== ONE-CLASS SVM CONFUSION MATRIX =====")
print(confusion_matrix(y_true, svm_y_pred))

===== MODEL PERFORMANCE =====


,Model,Accuracy,Precision,Recall,F1 Score
0,Isolation Forest,0.8700,0.4196,0.7833,0.5465
1,One-Class SVM,0.8917,0.4790,0.9500,0.6369



===== ISOLATION FOREST CONFUSION MATRIX =====
[[475  65]
 [ 13  47]]

===== ONE-CLASS SVM CONFUSION MATRIX =====
[[478  62]
 [  3  57]]


In [69]:
# ==========================================
# STEP 15: SPLIT SYNTHETIC ANOMALIES
# VALIDATION SET + FINAL TEST SET
# ==========================================

from sklearn.model_selection import train_test_split

synthetic_validation, synthetic_final_test = train_test_split(
    synthetic_anomalies,
    test_size=0.50,
    random_state=42,
    stratify=synthetic_anomalies["anomaly_type"]
)

print("Synthetic anomaly split completed successfully")

print("\nValidation anomalies:",
      len(synthetic_validation))

print("Final test anomalies:",
      len(synthetic_final_test))

print("\n===== VALIDATION ANOMALY TYPES =====")
print(
    synthetic_validation[
        "anomaly_type"
    ].value_counts()
)

print("\n===== FINAL TEST ANOMALY TYPES =====")
print(
    synthetic_final_test[
        "anomaly_type"
    ].value_counts()
)

Synthetic anomaly split completed successfully

Validation anomalies: 30
Final test anomalies: 30

===== VALIDATION ANOMALY TYPES =====
anomaly_type
weekend_gaming_high_usage    10
high_usage_low_sleep         10
night_social_low_sleep       10
Name: count, dtype: int64

===== FINAL TEST ANOMALY TYPES =====
anomaly_type
night_social_low_sleep       10
high_usage_low_sleep         10
weekend_gaming_high_usage    10
Name: count, dtype: int64


In [70]:
# ==========================================
# STEP 16: CREATE VALIDATION AND FINAL
# REFERENCE SETS
# ==========================================

reference_validation, reference_final_test = train_test_split(
    reference_data,
    test_size=0.50,
    random_state=42
)

# ------------------------------------------
# Build validation dataset
# ------------------------------------------

validation_data = pd.concat(
    [
        reference_validation,
        synthetic_validation
    ],
    axis=0
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# ------------------------------------------
# Build final test dataset
# ------------------------------------------

final_test_data = pd.concat(
    [
        reference_final_test,
        synthetic_final_test
    ],
    axis=0
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


print("Validation and final test datasets created successfully")

print("\n===== VALIDATION SET =====")
print("Total:", len(validation_data))
print(
    validation_data[
        "ground_truth_anomaly"
    ].value_counts().sort_index()
)

print("\n===== FINAL TEST SET =====")
print("Total:", len(final_test_data))
print(
    final_test_data[
        "ground_truth_anomaly"
    ].value_counts().sort_index()
)

# Check overlap between original reference records
overlap = set(reference_validation.index).intersection(
    set(reference_final_test.index)
)

print("\nReference record overlap:", len(overlap))

Validation and final test datasets created successfully

===== VALIDATION SET =====
Total: 300
ground_truth_anomaly
0    270
1     30
Name: count, dtype: int64

===== FINAL TEST SET =====
Total: 300
ground_truth_anomaly
0    270
1     30
Name: count, dtype: int64

Reference record overlap: 0


In [71]:
# ==========================================
# STEP 17: HYPERPARAMETER TUNING
# USING VALIDATION SET ONLY
# ==========================================

# Prepare validation features and labels
X_val = validation_data[behavioral_features].copy()

y_val = validation_data[
    "ground_truth_anomaly"
].values

# Use existing scaler
X_val_scaled = scaler.transform(X_val)


# ==========================================
# 1. ISOLATION FOREST TUNING
# ==========================================

iso_results = []

contamination_values = [
    0.03,
    0.05,
    0.08,
    0.10,
    0.12,
    0.15
]

for contamination in contamination_values:

    model = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=42
    )

    # Train ONLY on original training data
    model.fit(X_train_scaled)

    predictions = model.predict(X_val_scaled)

    y_pred = (predictions == -1).astype(int)

    precision = precision_score(
        y_val, y_pred, zero_division=0
    )

    recall = recall_score(
        y_val, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_val, y_pred, zero_division=0
    )

    iso_results.append({
        "contamination": contamination,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })


iso_results_df = pd.DataFrame(iso_results)

print("===== ISOLATION FOREST TUNING =====")

display(
    iso_results_df.round(4)
)


# ==========================================
# 2. ONE-CLASS SVM TUNING
# ==========================================

svm_results = []

nu_values = [
    0.03,
    0.05,
    0.08,
    0.10,
    0.12,
    0.15
]

gamma_values = [
    "scale",
    0.01,
    0.05,
    0.1
]

for nu in nu_values:

    for gamma in gamma_values:

        model = OneClassSVM(
            kernel="rbf",
            nu=nu,
            gamma=gamma
        )

        # Train ONLY on original training data
        model.fit(X_train_scaled)

        predictions = model.predict(X_val_scaled)

        y_pred = (
            predictions == -1
        ).astype(int)

        precision = precision_score(
            y_val, y_pred, zero_division=0
        )

        recall = recall_score(
            y_val, y_pred, zero_division=0
        )

        f1 = f1_score(
            y_val, y_pred, zero_division=0
        )

        svm_results.append({
            "nu": nu,
            "gamma": gamma,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })


svm_results_df = pd.DataFrame(
    svm_results
)

print("\n===== ONE-CLASS SVM TUNING =====")

display(
    svm_results_df.round(4)
)


# ==========================================
# 3. FIND BEST CONFIGURATIONS
# ==========================================

best_iso = iso_results_df.loc[
    iso_results_df["f1"].idxmax()
]

best_svm = svm_results_df.loc[
    svm_results_df["f1"].idxmax()
]


print("\n===== BEST ISOLATION FOREST =====")
print(best_iso)

print("\n===== BEST ONE-CLASS SVM =====")
print(best_svm)

===== ISOLATION FOREST TUNING =====


,contamination,precision,recall,f1
0,0.03,0.6774,0.7000,0.6885
1,0.05,0.5111,0.7667,0.6133
2,0.08,0.4694,0.7667,0.5823
3,0.10,0.4211,0.8000,0.5517
4,0.12,0.3906,0.8333,0.5319
5,0.15,0.3378,0.8333,0.4808



===== ONE-CLASS SVM TUNING =====


,nu,gamma,precision,recall,f1
0,0.03,scale,0.5778,0.8667,0.6933
1,0.03,0.01,0.7273,0.8000,0.7619
2,0.03,0.05,0.6154,0.8000,0.6957
3,0.03,0.1,0.5000,0.9000,0.6429
4,0.05,scale,0.5532,0.8667,0.6753
5,0.05,0.01,0.6098,0.8333,0.7042
6,0.05,0.05,0.5909,0.8667,0.7027
7,0.05,0.1,0.4912,0.9333,0.6437
8,0.08,scale,0.4667,0.9333,0.6222
9,0.08,0.01,0.5094,0.9000,0.6506



===== BEST ISOLATION FOREST =====
contamination    0.030000
precision        0.677419
recall           0.700000
f1               0.688525
Name: 0, dtype: float64

===== BEST ONE-CLASS SVM =====
nu               0.03
gamma            0.01
precision    0.727273
recall            0.8
f1           0.761905
Name: 1, dtype: object


In [72]:
# ==========================================
# STEP 18: FINAL MODEL EVALUATION
# ==========================================

# ------------------------------------------
# 1. Train tuned Isolation Forest
# ------------------------------------------

final_iso = IsolationForest(
    n_estimators=200,
    contamination=0.03,
    random_state=42
)

final_iso.fit(X_train_scaled)


# ------------------------------------------
# 2. Train tuned One-Class SVM
# ------------------------------------------

final_svm = OneClassSVM(
    kernel="rbf",
    nu=0.03,
    gamma=0.01
)

final_svm.fit(X_train_scaled)


# ------------------------------------------
# 3. Prepare untouched final test set
# ------------------------------------------

X_final = final_test_data[
    behavioral_features
].copy()

y_final = final_test_data[
    "ground_truth_anomaly"
].values

# IMPORTANT:
# Use existing training scaler only
X_final_scaled = scaler.transform(X_final)


# ------------------------------------------
# 4. Predictions
# ------------------------------------------

iso_final_raw = final_iso.predict(
    X_final_scaled
)

svm_final_raw = final_svm.predict(
    X_final_scaled
)

# Convert -1 anomaly -> 1
#          +1 normal  -> 0

iso_final_pred = (
    iso_final_raw == -1
).astype(int)

svm_final_pred = (
    svm_final_raw == -1
).astype(int)


# ------------------------------------------
# 5. Calculate final metrics
# ------------------------------------------

final_results = pd.DataFrame({
    "Model": [
        "Isolation Forest",
        "One-Class SVM"
    ],

    "Accuracy": [
        accuracy_score(
            y_final,
            iso_final_pred
        ),
        accuracy_score(
            y_final,
            svm_final_pred
        )
    ],

    "Precision": [
        precision_score(
            y_final,
            iso_final_pred,
            zero_division=0
        ),
        precision_score(
            y_final,
            svm_final_pred,
            zero_division=0
        )
    ],

    "Recall": [
        recall_score(
            y_final,
            iso_final_pred,
            zero_division=0
        ),
        recall_score(
            y_final,
            svm_final_pred,
            zero_division=0
        )
    ],

    "F1 Score": [
        f1_score(
            y_final,
            iso_final_pred,
            zero_division=0
        ),
        f1_score(
            y_final,
            svm_final_pred,
            zero_division=0
        )
    ]
})


print("===== FINAL TEST PERFORMANCE =====")

display(
    final_results.round(4)
)


# ------------------------------------------
# 6. Final confusion matrices
# ------------------------------------------

print("\n===== ISOLATION FOREST =====")
print(
    confusion_matrix(
        y_final,
        iso_final_pred
    )
)

print("\n===== ONE-CLASS SVM =====")
print(
    confusion_matrix(
        y_final,
        svm_final_pred
    )
)

===== FINAL TEST PERFORMANCE =====


,Model,Accuracy,Precision,Recall,F1 Score
0,Isolation Forest,0.9133,0.5833,0.4667,0.5185
1,One-Class SVM,0.9467,0.7692,0.6667,0.7143



===== ISOLATION FOREST =====
[[260  10]
 [ 16  14]]

===== ONE-CLASS SVM =====
[[264   6]
 [ 10  20]]


In [73]:
import joblib

joblib.dump(final_svm, "behavioral_anomaly_ocsvm.pkl")
joblib.dump(scaler, "behavioral_scaler.pkl")
joblib.dump(behavioral_features, "behavioral_features.pkl")

print("Final model artifacts saved successfully")

Final model artifacts saved successfully


In [74]:
from google.colab import files

files.download("behavioral_anomaly_ocsvm.pkl")
files.download("behavioral_scaler.pkl")
files.download("behavioral_features.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>